# 全球標準來源全量核查
保存本輪實際腳本；取得脚本禁止覆寫基線，重跑須另立批次。來源原文、中譯參考與未匹配狀態分列。

In [ ]:
from pathlib import Path
__file__ = str(Path.cwd() / '取得與核查.py')
from pathlib import Path
from urllib.request import Request,urlopen
from datetime import datetime,timezone
from collections import Counter,defaultdict
import csv,io,json,hashlib,xml.etree.ElementTree as ET,subprocess
工作=Path(__file__).resolve().parent;根=工作.parents[2];H=lambda b:hashlib.sha256(b).hexdigest()
原=工作/'原始來源';原.mkdir(exist_ok=True)
表=根/'Reference/tables/93_全球貨幣與市場代碼全量續核_B71_20261012';表.mkdir(parents=True,exist_ok=True)
assert not (工作/'基線.json').exists(),'不得刷新基線'
paths=subprocess.check_output(['git','ls-files','-z'],cwd=根).decode().split('\0')
(工作/'基線.json').write_text(json.dumps({'提交':subprocess.check_output(['git','rev-parse','HEAD'],cwd=根).decode().strip(),'既有正文':{p:H((根/p).read_bytes()) for p in paths if p.endswith(('.md','.qmd')) and (根/p).is_file()}},ensure_ascii=False,indent=2),encoding='utf-8')
收=[]
def 取(名,url):
    with urlopen(Request(url,headers={'User-Agent':'Global-public-registry-audit/1.0'}),timeout=60) as r:
        b=r.read(15*1024*1024+1);assert len(b)<=15*1024*1024
        p=原/名;p.write_bytes(b)
        收.append({'中文來源':名,'網址':url,'觀測UTC':datetime.now(timezone.utc).isoformat(),'狀態碼':r.status,'回應標頭':dict(r.headers),'原始檔':p.relative_to(根).as_posix(),'來源雜湊':H(b),'實得位元組':len(b)})
    return b
def 讀(p):
    with (根/p).open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
def 寫(n,rows):
    assert rows
    with (表/n).open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
母=讀('Reference/tables/01_country_area/registry_country_area_iso3166_m49_e164_20261004.csv');assert len({r['iso_alpha2'] for r in 母})==len(母)
舊幣=讀('Reference/tables/21_currency_broker_b03_20261007/registry_iso4217_all_entries.csv')
舊市=讀('Reference/tables/24_financial_universe_b06_20261007/registry_iso10383_mic.csv')
別=defaultdict(set)
for r in 舊幣:
    if r['iso_alpha2']:別[r['source_entity_name']].add(r['iso_alpha2'])
for r in 母:
    別[r['name_en'].upper()].add(r['iso_alpha2'])
    if r['official_name_en']:別[r['official_name_en'].upper()].add(r['iso_alpha2'])
領=json.loads(取('中文區域名稱參考.json','https://raw.githubusercontent.com/unicode-org/cldr-json/main/cldr-json/cldr-localenames-full/main/zh-Hant/territories.json'))['main']['zh-Hant']['localeDisplayNames']['territories']
幣譯=json.loads(取('中文貨幣名稱參考.json','https://raw.githubusercontent.com/unicode-org/cldr-json/main/cldr-json/cldr-numbers-full/main/zh-Hant/currencies.json'))['main']['zh-Hant']['numbers']['currencies']
幣=[];版={}
for state,tag,name in [('CURRENT','CcyNtry','現行貨幣與基金.xml'),('HISTORICAL','HstrcCcyNtry','歷史貨幣與基金.xml')]:
    url='https://www.six-group.com/dam/download/financial-information/data-center/iso-currrency/lists/'+('list-one.xml' if state=='CURRENT' else 'list-three.xml')
    x=ET.fromstring(取(name,url));版[state]=dict(x.attrib)
    entries=x.findall('.//'+tag);assert entries
    for i,e in enumerate(entries,1):
        entity=e.findtext('CtryNm','');code=e.findtext('Ccy','');a=別.get(entity,set());iso=next(iter(a)) if len(a)==1 else ''
        幣.append({'記錄識別碼':state+':'+str(i),'狀態中譯':'現行' if state=='CURRENT' else '歷史','來源實體名稱原值':entity,'國家地區ISO代碼':iso,'國家地區中譯參考':領.get(iso,'待核實'),'關聯證據':'沿用既有名稱對照，未提升為官方代碼關聯' if iso else '待核實；保留來源實體','貨幣名稱原值':e.findtext('CcyNm',''),'貨幣中譯參考':幣譯.get(code,{}).get('displayName','待核實'),'字母代碼':code,'數字代碼':e.findtext('CcyNbr',''),'最小單位原值':e.findtext('CcyMnrUnts',''),'撤回日期原值':e.findtext('WthdrwlDt',''),'基金屬性原值':json.dumps(e.find('CcyNm').attrib if e.find('CcyNm') is not None else {},ensure_ascii=False),'來源XML':ET.tostring(e,encoding='unicode')})
    assert len([r for r in 幣 if r['狀態中譯']==('現行' if state=='CURRENT' else '歷史')])==len(entries)
取('市場代碼官方發布頁.html','https://www.iso20022.org/market-identifier-codes')
mic=取('完整市場識別碼.csv','https://www.iso20022.org/sites/default/files/ISO10383_MIC/ISO10383_MIC.csv')
reader=csv.DictReader(io.StringIO(mic.decode('utf-8-sig')));市=[]
for i,r in enumerate(reader,1):
    code=r['MIC'];country=r['ISO COUNTRY CODE (ISO 3166)']
    市.append({'来源序號':i,'市場識別碼':code,'營運市場識別碼':r['OPERATING MIC'],'營運或分段原值':r['OPRT/SGMT'],'市場名稱原值':r['MARKET NAME-INSTITUTION DESCRIPTION'],'市場名稱中譯狀態':'待逐項校訂；原名稱保留','法定實體名稱原值':r['LEGAL ENTITY NAME'],'法定實體中譯狀態':'待逐項校訂；不得猜譯','國家地區ISO代碼':country,'國家地區中譯參考':領.get(country,'待核實'),'狀態原值':r['STATUS'],'分類原值':r['MARKET CATEGORY CODE'],'網站原值':r['WEBSITE'],'來源所有欄位JSON':json.dumps(r,ensure_ascii=False,sort_keys=True)})
assert 市 and len({r['市場識別碼'] for r in 市})==len(市)
寫('全量現行與歷史貨幣.csv',幣);寫('全量市場識別碼.csv',市)
覆=[]
for m in 母:
    c=m['iso_alpha2'];cc=[r for r in 幣 if r['國家地區ISO代碼']==c];mm=[r for r in 市 if r['國家地區ISO代碼']==c]
    覆.append({'國家地區ISO代碼':c,'國家地區中譯參考':領.get(c,'待核實'),'名稱原值':m['name_en'],'現行貨幣來源行':sum(r['狀態中譯']=='現行' for r in cc),'歷史貨幣來源行':sum(r['狀態中譯']=='歷史' for r in cc),'市場代碼來源行':len(mm),'上市公司全量狀態':'UNKNOWN（未完成）','券商全量狀態':'UNKNOWN（未完成）','所有貨幣全量狀態':'UNKNOWN（ISO登記表已接收；地方及更早歷史貨幣未竣工）','證據邊界':'無來源行不等於沒有貨幣或市場；市場代碼不等於公司或券商'})
寫('逐國家地區覆蓋與缺口.csv',覆)
舊碼={r['MIC']:r for r in 舊市};新碼={r['市場識別碼']:r for r in 市}
差=[{'市場識別碼':c,'變更類型':'新增代碼' if c not in 舊碼 else '本次未見舊代碼','舊狀態':舊碼.get(c,{}).get('status',''),'新狀態':新碼.get(c,{}).get('狀態原值','')} for c in sorted(set(舊碼)^set(新碼))]
for c in sorted(set(舊碼)&set(新碼)):
    if 舊碼[c]['status']!=新碼[c]['狀態原值']:差.append({'市場識別碼':c,'變更類型':'狀態改變','舊狀態':舊碼[c]['status'],'新狀態':新碼[c]['狀態原值']})
if 差:寫('市場代碼跨批次差異.csv',差)
結={'核查UTC':datetime.now(timezone.utc).isoformat(),'國家地區母表行數':len(母),'現行貨幣來源行':sum(r['狀態中譯']=='現行' for r in 幣),'歷史貨幣來源行':sum(r['狀態中譯']=='歷史' for r in 幣),'貨幣來源實體未關聯行':sum(not r['國家地區ISO代碼'] for r in 幣),'現行唯一非空字母代碼':len({r['字母代碼'] for r in 幣 if r['狀態中譯']=='現行' and r['字母代碼']}),'貨幣源版本':版,'市场代碼來源行':len(市),'市場國家地區代碼數':len({r['國家地區ISO代碼'] for r in 市}),'市場狀態分布':dict(Counter(r['狀態原值'] for r in 市)),'市場跨批次差異項':len(差),'官方及中譯參考回應數':len(收),'全球上市公司已齊':False,'全球券商已齊':False,'所有歷史貨幣已齊':False}
(工作/'來源收據.json').write_text(json.dumps(收,ensure_ascii=False,indent=2),encoding='utf-8')
(工作/'核查結果.json').write_text(json.dumps(結,ensure_ascii=False,indent=2),encoding='utf-8');print(json.dumps(結,ensure_ascii=False),flush=True)


In [ ]:
from pathlib import Path
__file__ = str(Path.cwd() / '差異與交付.py')
from pathlib import Path
from collections import Counter
import csv,json,hashlib,xml.etree.ElementTree as ET
工作=Path(__file__).resolve().parent;根=工作.parents[2];表=根/'Reference/tables/93_全球貨幣與市場代碼全量續核_B71_20261012';H=lambda b:hashlib.sha256(b).hexdigest()
基=json.loads((工作/'基線.json').read_text(encoding='utf-8'));assert all(H((根/p).read_bytes())==h for p,h in 基['既有正文'].items())
def 讀(p):
    with p.open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
def 寫(n,rows,fields):
    with (表/n).open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows)
幣=讀(表/'全量現行與歷史貨幣.csv');市=讀(表/'全量市場識別碼.csv');未=[r for r in 幣 if not r['國家地區ISO代碼']]
寫('未關聯貨幣來源記錄.csv',未,list(幣[0]))
舊市={r['MIC']:json.loads(r['raw_fields_json']) for r in 讀(根/'Reference/tables/24_financial_universe_b06_20261007/registry_iso10383_mic.csv')};差=[]
for r in 市:
    c=r['市場識別碼'];n=json.loads(r['來源所有欄位JSON']);o=舊市.get(c,{})
    keys=sorted(k for k in set(n)|set(o) if n.get(k)!=o.get(k))
    if keys:差.append({'市場識別碼':c,'差異欄位原名':' / '.join(keys),'舊所有欄位JSON':json.dumps(o,ensure_ascii=False),'新所有欄位JSON':json.dumps(n,ensure_ascii=False)})
寫('市場全部欄位跨批次差異.csv',差,['市場識別碼','差異欄位原名','舊所有欄位JSON','新所有欄位JSON'])
def 特徵(state,xml):
    x=ET.fromstring(xml)
    return (state,tuple((e.tag,e.text or '',tuple(sorted(e.attrib.items()))) for e in x))
舊幣=讀(根/'Reference/tables/21_currency_broker_b03_20261007/registry_iso4217_all_entries.csv')
a=Counter(特徵(r['state'],r['raw_xml']) for r in 舊幣);b=Counter(特徵('CURRENT' if r['狀態中譯']=='現行' else 'HISTORICAL',r['來源XML']) for r in 幣)
結=json.loads((工作/'核查結果.json').read_text(encoding='utf-8'));結.update({'市場全部欄位差異記錄數':len(差),'未關聯貨幣實體名稱數':len({r['來源實體名稱原值'] for r in 未}),'貨幣來源多重集合新增行':sum((b-a).values()),'貨幣來源多重集合未見舊行':sum((a-b).values()),'市場營運分段分布':dict(Counter(r['營運或分段原值'] for r in 市))})
(工作/'核查結果.json').write_text(json.dumps(結,ensure_ascii=False,indent=2),encoding='utf-8')
報='''---
title: "第七十一批：全球貨幣與市場識別碼全量續核"
lang: zh-TW
format: html
---

本輪核對最新提交及接續清單，轉回全球金融資料。另存官方完整下載，不覆寫舊資料庫、前批清單或使用者報告。國家／地區母表取現有完整表，實際讀取行數，不把249設定為歷史政治實體的總量上限。

## 完整來源接收與對賬

'''+f"- 現有國家／地區母表：{結['國家地區母表行數']} 行，全部納入本輪覆蓋表。\n- 現行貨幣與基金：{結['現行貨幣來源行']} 行；唯一非空字母代碼 {結['現行唯一非空字母代碼']} 個。\n- 歷史貨幣與基金：{結['歷史貨幣來源行']} 行。\n- 市場識別碼：{結['市场代碼來源行']:,} 行，涉及 {結['市場國家地區代碼數']} 個來源國家／地區代碼。\n- 未可靠關聯至母表的貨幣來源記錄：{結['貨幣來源實體未關聯行']} 行、{結['未關聯貨幣實體名稱數']} 個來源名稱，全部另表保留。\n\n"+'''[SIX（瑞士金融基礎設施集團）維護的ISO 4217貨幣表](https://www.six-group.com/en/products-services/financial-information/market-reference-data/data-standards.html)原始版本日期：現行表2026年9月17日、歷史表2026年1月1日。本次取得日期是2026年10月12日；取得日期不取代來源發布日期。

[ISO 10383市場識別碼官方發布頁](https://www.iso20022.org/market-identifier-codes)本次仍展示2026年9月14日發布、9月28日實施、10月12日下次發布的時程；不因今日是預定發布日，就宣稱10月份版本已取得。

'''+f"市場狀態原值分布：ACTIVE（來源標記有效）{結['市場狀態分布'].get('ACTIVE',0)}、EXPIRED（來源標記失效）{結['市場狀態分布'].get('EXPIRED',0)}、UPDATED（來源標記更新）{結['市場狀態分布'].get('UPDATED',0)}。這些狀態不是本輪實測營業或監管資格。\n\n市場全部來源欄位與既有第六批對照，差異 {結['市場全部欄位差異記錄數']} 條；貨幣來源多重集合與原第三批對照，新增 {結['貨幣來源多重集合新增行']} 行、未見舊行 {結['貨幣來源多重集合未見舊行']} 行。零差異只表示這次源表與指定基線一致，不是全球金融活動沒有變動。\n\n"+'''## 業務定義與完整性邊界

貨幣表的一行是「來源實體—貨幣／基金」關係，同一代碼可以出現多次；字母代碼包括基金、貴金屬、測試或無貨幣等用途，不能當作日常流通貨幣種類總數。ISO歷史表也不等於人類文明所有地方、古代及非ISO貨幣。

MIC（市場識別碼）識別交易場所、交易平台、分段或報告設施；營運代碼與分段代碼不等於不同公司，更不等於證券行或上市公司總数。保留有效、失效及更新狀態，不刪歷史行。來源未列市場的國家／地區不推定沒有金融市場。

貨幣名稱關聯沿用既有對照，不把編輯對照升级為ISO官方逐國關聯。未匹配行完整保留。中文區域與貨幣名称參考 [Unicode CLDR（通用區域資料庫）](https://github.com/unicode-org/cldr-json)，原名稱並存；市場與法定實體名稱中譯仍逐項待校，不編造中譯或取代法定原名。

## 尚未竣工與接續施工

逐國覆蓋表保留全部母表行及上市公司、券商、所有貨幣的待完成狀態；本輪沒有宣稱收齊每家公司。下一步優先逐項校核未關聯來源名稱、歷史政治實體與基金專用實體的區別，再按實際交易所及監管名冊補齊公司與持牌券商，對賬來源總量和分頁，不以市場識別碼代替名冊。

逐用途授權、全球各家公司的完整性及人類歷史所有貨幣仍待核實。完整原始回應、取得時間、標頭及SHA256（安全雜湊算法）保存在 `reports/2026-10-12/全球貨幣與市場代碼全量續核_B71/`；所有來源欄位及逐國覆蓋見 `tables/93_全球貨幣與市場代碼全量續核_B71_20261012/`。
'''
(根/'Reference/全球貨幣與市場代碼全量續核_B71_20261012.qmd').write_text(報,encoding='utf-8')
主='Reference/Geostrategy_Defense_Industry_Aerospace_and_Artificial_Interlligence_Ecosystem_Report.qmd';原=(根/主).read_bytes()
附=('\n\n## 第七十一批有界增補：全球金融標準母表續核\n\n全量接收280條現行貨幣與基金、169條歷史記錄、2,883條市場識別碼，全部249行現有國家／地區母表納入覆蓋表。42條貨幣來源記錄尚未可靠關聯，另表完整保留。市場代碼不等於上市公司或券商；歷史ISO貨幣表不等於所有歷史貨幣。來源版本與取得日期分列，未宣稱全球已竣工。詳見[第七十一批報告](全球貨幣與市場代碼全量續核_B71_20261012.qmd)。\n').encode('utf-8')
(根/主).write_bytes(原+附)
(工作/'附加收據.json').write_text(json.dumps({'主報告':主,'原文位元組':len(原),'原文雜湊':H(原),'附加雜湊':H(附)},ensure_ascii=False,indent=2),encoding='utf-8')
for d,t in {'redteam':'市場代碼、分段與法定實體不能混成券商或上市公司總數。','critic':'完整接收標準表不等於收齊全球名冊；全部249行覆蓋表保留待完成狀態。','killcritic':'全來源欄位、多重集合對照與原始雜湊提供可重現證據；不隱藏零差異或未匹配行。','blindspot':'42條未關聯來源記錄可能涉及歷史政治實體、基金專用名稱或名稱差異，須逐項核實。','blueprint':'來源實體、貨幣代碼、交易場所、法定公司及監管資格分表，透過有證據的關聯連接。','cheatsheet':'280條現行來源關係、169條歷史關係、2,883條MIC；不直接當作貨幣種類或公司數。','actionplan':'先核實35個未匹配來源名稱，再逐交易所／監管名冊全量對賬公司與券商；保留全部國家／地區工作。'}.items():
    (根/d/'全球貨幣與市場代碼全量續核_B71_20261012.md').write_text('# 第七十一批核查筆記\n\n'+t+'\n\n[報告](../Reference/全球貨幣與市場代碼全量續核_B71_20261012.qmd)。\n',encoding='utf-8')
print(json.dumps(結,ensure_ascii=False))


In [ ]:
from pathlib import Path
__file__ = str(Path.cwd() / '驗收.py')
from pathlib import Path
from collections import Counter
import json,csv,hashlib,io,xml.etree.ElementTree as ET
工作=Path(__file__).resolve().parent;根=工作.parents[2];H=lambda b:hashlib.sha256(b).hexdigest()
基=json.loads((工作/'基線.json').read_text(encoding='utf-8'));附=json.loads((工作/'附加收據.json').read_text(encoding='utf-8'))
for p,h in 基['既有正文'].items():
    b=(根/p).read_bytes()
    if p==附['主報告']:
        n=附['原文位元組'];assert H(b[:n])==h==附['原文雜湊'] and H(b[n:])==附['附加雜湊']
    else:assert H(b)==h,'既有正文變更：'+p
收=json.loads((工作/'來源收據.json').read_text(encoding='utf-8'));結=json.loads((工作/'核查結果.json').read_text(encoding='utf-8'));assert len(收)==6
for r in 收:
    b=(根/r['原始檔']).read_bytes();assert H(b)==r['來源雜湊'] and len(b)==r['實得位元組'] and r['狀態碼']==200
表=根/'Reference/tables/93_全球貨幣與市場代碼全量續核_B71_20261012'
def 讀(p):
    with p.open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
幣=讀(表/'全量現行與歷史貨幣.csv');市=讀(表/'全量市場識別碼.csv');覆=讀(表/'逐國家地區覆蓋與缺口.csv');未=讀(表/'未關聯貨幣來源記錄.csv')
母=讀(根/'Reference/tables/01_country_area/registry_country_area_iso3166_m49_e164_20261004.csv');codes={r['iso_alpha2'] for r in 母}
assert len(覆)==len(母)==結['國家地區母表行數'] and {r['國家地區ISO代碼'] for r in 覆}==codes
for name,tag,state in [('現行貨幣與基金.xml','CcyNtry','現行'),('歷史貨幣與基金.xml','HstrcCcyNtry','歷史')]:
    e=ET.parse(工作/'原始來源'/name).findall('.//'+tag);rows=[r for r in 幣 if r['狀態中譯']==state]
    assert len(e)==len(rows)
    for x,r in zip(e,rows):
        y=ET.fromstring(r['來源XML']);assert ET.tostring(x).strip()==ET.tostring(y).strip()
        assert r['字母代碼']==x.findtext('Ccy','') and r['來源實體名稱原值']==x.findtext('CtryNm','')
src=讀(工作/'原始來源/完整市場識別碼.csv');assert len(src)==len(市)==結['市场代碼來源行']
assert all(a==json.loads(b['來源所有欄位JSON']) for a,b in zip(src,市))
assert len({r['市場識別碼'] for r in 市})==len(市)
assert len(未)==結['貨幣來源實體未關聯行']==sum(not r['國家地區ISO代碼'] for r in 幣)
assert {r['國家地區ISO代碼'] for r in 幣 if r['國家地區ISO代碼']}<=codes
for c in 覆:
    iso=c['國家地區ISO代碼']
    assert int(c['現行貨幣來源行'])==sum(r['國家地區ISO代碼']==iso and r['狀態中譯']=='現行' for r in 幣)
    assert int(c['歷史貨幣來源行'])==sum(r['國家地區ISO代碼']==iso and r['狀態中譯']=='歷史' for r in 幣)
    assert int(c['市場代碼來源行'])==sum(r['國家地區ISO代碼']==iso for r in 市)
assert not 結['全球上市公司已齊'] and not 結['全球券商已齊'] and not 結['所有歷史貨幣已齊']
html=根/'Reference/全球貨幣與市場代碼全量續核_B71_20261012.html';assert html.stat().st_size>1000
(工作/'交付驗收.json').write_text(json.dumps({'既有正文保留數':len(基['既有正文']),'主報告原文保留':True,'來源回應雜湊核對數':6,'母表全部行納入':True,'貨幣原始元素逐項對賬':True,'市場所有欄位逐項對賬':True,'未關聯行保留':True,'成品雜湊':H(html.read_bytes()),'市場來源未落母表行數':sum(r['國家地區ISO代碼'] not in codes for r in 市)},ensure_ascii=False,indent=2),encoding='utf-8')
print('全球來源逐項對賬、母表覆蓋及原文保留驗收通過。')
